In [ ]:
import pandas as pd
import numpy as np
import ast

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity



In [ ]:
!pip install scikit-surprise --no-cache-dir

In [ ]:
!pip install "numpy<2.0"

In [ ]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split

In [ ]:
from google.colab import files
files.upload()

In [ ]:
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")
ratings = pd.read_csv("ratings.csv")

In [ ]:
movies = movies.merge(credits, on='title')

In [ ]:
movies.dropna(inplace=True)

In [ ]:
movies['genres'].head()


In [ ]:
movies['genres'].iloc[1]

'[{"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 28, "name": "Action"}]'

In [ ]:
movies['keywords'].head()


In [ ]:
#exacts the name only from generes and keywords


def convert(text):
    # If already a list → return directly
    if isinstance(text, list):
        return text

    # If string → convert
    try:
        return [i['name'] for i in ast.literal_eval(text)]
    except:
        return []

In [ ]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

In [ ]:
movies['keywords'].head()


In [ ]:


def convert_cast(text):
    # If already list → take first 3 actors
    if isinstance(text, list):
        return text[:3]

    # If string → parse it
    try:
        L = []
        for i in ast.literal_eval(text):
            L.append(i['name'])
        return L[:3]
    except:
        return []

import ast

def fetch_director(text):
    # Case 1: Already a list
    if isinstance(text, list):
        # Check if it's list of dicts OR list of names
        if len(text) > 0 and isinstance(text[0], dict):
            return [i['name'] for i in text if i.get('job') == 'Director']
        else:
            # Already names → just return
            return text

    # Case 2: String → parse it
    try:
        L = []
        for i in ast.literal_eval(text):
            if i['job'] == 'Director':
                L.append(i['name'])
        return L
    except:
        return []

In [ ]:
movies['cast'] = movies['cast'].apply(convert_cast)
movies['crew'] = movies['crew'].apply(fetch_director)
movies['overview'] = movies['overview'].apply(lambda x:x.split())


In [ ]:
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [ ]:
movies

In [ ]:
def collapse(L):
    L1 = []
    for i in L:
        L1.append(i.replace(" ",""))
    return L1

In [ ]:
movies['cast'] = movies['cast'].apply(collapse)
movies['crew'] = movies['crew'].apply(collapse)
movies['genres'] = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)

In [ ]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']
new_df= movies.drop(columns=['overview','genres','keywords','cast','crew'])

In [ ]:
movies


In [ ]:
new_df

In [ ]:
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

In [ ]:
new_df

In [ ]:
# CONTENT-BASED MODEL
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()

cosine_sim = cosine_similarity(vectors)


In [ ]:
#SVD MODEL (COLLABORATIVE FILTERING)
#“Ratings in my dataset range from 1 to 5”
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(
    ratings[['userId', 'movieId', 'rating']], reader
)
'''
Takes your pandas DataFrame (ratings)
Keeps only 3 columns:
userId
movieId
rating
Converts it into Surprise dataset format
'''

'\nTakes your pandas DataFrame (ratings)\nKeeps only 3 columns:\nuserId\nmovieId\nrating\nConverts it into Surprise dataset format\n'

In [ ]:
data

In [ ]:
trainset, testset = train_test_split(data, test_size=0.2)

svd_model = SVD()
svd_model.fit(trainset)


In [ ]:
def hybrid_recommend(user_id, movie_name, top_n=5):

    if movie_name not in new_df['title'].values:
        print(f"Movie '{movie_name}' not found in the dataset.")
        return []
    movie_index = new_df[new_df['title'] == movie_name].index[0]

    distances = cosine_sim[movie_index]

    movie_list = sorted(
        #enumerate adds index to the distances
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:30]

    recommendations = []
    #i(index ,score)
    for i in movie_list:
        idx = i[0]
        title = new_df.iloc[idx]['title']
        movie_id = new_df.iloc[idx]['movie_id']

        try:
            pred_rating = svd_model.predict(user_id, movie_id).est
        except:
            pred_rating = 0

        content_score = i[1]
        final_score = (0.6 * content_score) + (0.4 * pred_rating)

        # ✅ ALWAYS tuple
        recommendations.append((title, final_score))
    print("\n🎬 Recommended Movies:\n")
    return recommendations[:top_n]

In [ ]:
user_id = 1
movie_name = "Titanic"

results = hybrid_recommend(user_id, movie_name,5)


for movie, score in results:
    print(f"{movie}  (Score: {round(score,2)})")


🎬 Recommended Movies:

Spectre  (Score: 1.87)
Never Say Never Again  (Score: 1.85)
Die Another Day  (Score: 1.82)
Octopussy  (Score: 1.85)
Quantum of Solace  (Score: 1.81)


In [ ]:
movies

In [ ]:
print(new_df.columns)

Index(['movie_id', 'title', 'tags'], dtype='object')
